In [24]:
import mlflow
import pandas as pd
import time
import logging
import nltk
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

In [11]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
227,Greetings again from the darkness. Director Al...,positive
217,Historical movies always take liberties -- con...,negative
61,I am in a movie club at my school and I was fo...,negative
591,"The best thing about the movie is the name, as...",negative
883,In Holland a gay writer Gerard (Jeroen Krabbe)...,positive


In [12]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [25]:
#import nltk
nltk.download('stopwords')
nltk.download('wordnet')
df = normalize_text(df)
df.head()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\6035s\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\6035s\AppData\Roaming\nltk_data...


,review,sentiment
227,greeting darkness director alejandro amenabar ...,positive
217,historical movie always take liberty conversat...,negative
61,movie club school forced sit watch utterly dis...,negative
591,best thing movie name describes plot acting le...,negative
883,holland gay writer gerard jeroen krabbe give l...,positive


In [26]:
df['sentiment'].value_counts()

sentiment
positive    252
negative    248
Name: count, dtype: int64

In [27]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [28]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
227,greeting darkness director alejandro amenabar ...,1
217,historical movie always take liberty conversat...,0
61,movie club school forced sit watch utterly dis...,0
591,best thing movie name describes plot acting le...,0
883,holland gay writer gerard jeroen krabbe give l...,1


In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 500 entries, 227 to 512
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     500 non-null    object
 1   sentiment  500 non-null    int64 
dtypes: int64(1), object(1)
memory usage: 11.7+ KB


In [30]:
df.describe()

,sentiment
count,500.000000
mean,0.504000
std,0.500485
min,0.000000
25%,0.000000
50%,1.000000
75%,1.000000
max,1.000000


In [31]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [33]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [34]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [35]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/Shivam6035/Capstone-project-MLOps.mlflow')
dagshub.init(repo_owner='shivam6035', repo_name='Capstone-Project-Mlops', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")


2026-06-03 01:06:30,076 - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/shivam6035/Capstone-Project-Mlops "HTTP/1.1 200 OK"


Initialized MLflow to track repo "shivam6035/Capstone-Project-Mlops"

2026-06-03 01:06:30,081 - INFO - Initialized MLflow to track repo "shivam6035/Capstone-Project-Mlops"


Repository shivam6035/Capstone-Project-Mlops initialized!

2026-06-03 01:06:30,083 - INFO - Repository shivam6035/Capstone-Project-Mlops initialized!


<Experiment: artifact_location='mlflow-artifacts:/078475a7074b42c5af461142490831db', creation_time=1780428327246, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1780428327246, lifecycle_stage='active', name='Logistic Regression Baseline', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [36]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 100)
        mlflow.log_param("test_size", 0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

#         Save#  and log the notebook
#         notebook_path = "exp1_baseline_model.ipynb"
# #          logging.info("Executing Jupyter Notebook. This may take a while...")
#         os# .system(f"jupyter nbconvert --to notebook --# execute --inplace {notebook_path}")
#         mlflow.log_artifact(notebook_path)

        logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-06-03 01:06:43,230 - INFO - Starting MLflow run...
2026-06-03 01:06:43,742 - INFO - Logging preprocessing parameters...
2026-06-03 01:06:44,884 - INFO - Initializing Logistic Regression model...
2026-06-03 01:06:44,885 - INFO - Fitting the model...
2026-06-03 01:06:44,913 - INFO - Model training complete.
2026-06-03 01:06:44,914 - INFO - Logging model parameters...
2026-06-03 01:06:45,268 - INFO - Making predictions...
2026-06-03 01:06:45,269 - INFO - Calculating evaluation metrics...
2026-06-03 01:06:45,278 - INFO - Logging evaluation metrics...
2026-06-03 01:06:46,710 - INFO - Saving and logging the model...
2026/06/03 01:06:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/03 01:06:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The rec

🏃 View run marvelous-bat-906 at: https://dagshub.com/Shivam6035/Capstone-project-MLOps.mlflow/#/experiments/0/runs/5877c6f29cd94a94b676270a6979864a
🧪 View experiment at: https://dagshub.com/Shivam6035/Capstone-project-MLOps.mlflow/#/experiments/0
